<a href="https://colab.research.google.com/github/BaseballCV/BaseballCV/blob/main/notebooks/base_detector_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BaseballCV — Base Detector Example

This notebook demonstrates the **base detector** integration end-to-end:

- `base_detector` model (YOLOv11s, detects bases in broadcast feeds)
- `base_detect_dataset` (YOLO-format dataset)
- `base_detector_coco` (COCO-format dataset)

All three are resolved through `LoadTools` aliases and downloaded from the BallDataLab vault. Section 5 lets you run the detector on your own footage.

The dataset was annotated and the model trained by **Evan Arey** (see the repo README for contact).

> **Note:** requires `baseballcv` 0.1.25 or later ([PR #179](https://github.com/BaseballCV/BaseballCV/pull/179)).

**Runtime:** `Runtime → Change runtime type → GPU` recommended (CPU works, just slower).

In [ ]:
# Install BaseballCV (base detector requires >= 0.1.25; 0.1.26+ installs on current Colab Python)
!pip install -q "baseballcv>=0.1.26"

In [ ]:
import os
import json
import urllib.request
from collections import Counter

import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

from baseballcv.functions import LoadTools

load_tools = LoadTools()
print("Setup OK")

## 1. Load the model

`load_model` resolves the alias through the BallDataLab API (`balldatalab.com/api/models/base_detector`), which redirects to the vault-hosted weights.

In [ ]:
model_path = load_tools.load_model('base_detector')
print("Model weights at:", model_path)

model = YOLO(model_path)
print("Task:", model.task, "| Classes:", model.names)

## 2. Inference on a broadcast example

A real broadcast frame from the repo's assets. Note that this close-up angle is far from the training distribution, so we use a low confidence threshold here to give the model a chance — for typical wide broadcast frames, use conf ≥ 0.5 (see section 5 for why).

In [ ]:
IMG_URL = "https://raw.githubusercontent.com/BaseballCV/BaseballCV/main/assets/Tatis_SB.jpg"
img_path = "example.jpg"
urllib.request.urlretrieve(IMG_URL, img_path)

result = model.predict(img_path, conf=0.1, verbose=False)[0]

plt.figure(figsize=(12, 6))
plt.imshow(result.plot()[:, :, ::-1])  # plot() returns BGR
plt.axis("off")
plt.title(f"base_detector — {len(result.boxes)} detection(s) at conf >= 0.1")
plt.show()

## 3. COCO dataset — `base_detector_coco`

Downloads and extracts the COCO-format dataset (~150 MB): 660 images, 668 annotations, single class `base` (id `0`).

In [ ]:
coco_dir = load_tools.load_dataset('base_detector_coco')
print("Dataset extracted to:", coco_dir)

In [ ]:
ann_file = os.path.join(str(coco_dir), "annotations", "instances_default.json")
with open(ann_file) as f:
    coco = json.load(f)

cats = {c["id"]: c["name"] for c in coco["categories"]}
dist = Counter(cats[a["category_id"]] for a in coco["annotations"])

print("Images:", len(coco["images"]))
print("Annotations:", len(coco["annotations"]))
print("Categories:", cats)
print("Class distribution:", dict(dist))

### Ground truth vs. model prediction

Side-by-side on a random annotated image (the model was trained on these, so detections should match well here — this is a pipeline test, not a benchmark).

In [ ]:
# CVAT exports put images under images/default/ while file_name holds only the basename,
# so index every image under images/ by filename instead of assuming a flat folder.
image_paths = {f: os.path.join(root, f)
               for root, _, fs in os.walk(os.path.join(str(coco_dir), "images")) for f in fs}

img_info = next(im for im in coco["images"]
                if any(a["image_id"] == im["id"] for a in coco["annotations"]))
sample_path = image_paths[os.path.basename(img_info["file_name"])]
gt = [a for a in coco["annotations"] if a["image_id"] == img_info["id"]]

pred = model.predict(sample_path, conf=0.25, verbose=False)[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(Image.open(sample_path))
axes[0].set_title(f"Ground truth — {len(gt)} box(es)")
for a in gt:
    x, y, w, h = a["bbox"]
    axes[0].add_patch(plt.Rectangle((x, y), w, h, fill=False, color="lime", linewidth=2))
axes[1].imshow(pred.plot()[:, :, ::-1])
axes[1].set_title(f"base_detector — {len(pred.boxes)} detection(s)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. YOLO dataset — `base_detect_dataset`

Same 660 images in YOLO format. Known issue (documented in the dataset README): 107 of the 660 images have no label file yet, and there is no val/test split.

## 5. Run it on your own footage

Upload an image or video (MP4) and run the detector on it. On Colab, uncomment the `files.upload()` line to get an upload prompt, or point `INPUT_PATH` at any URL / local path.

**Known limitation:** the model was trained on labeled bases only, so it can false-positive on other white field objects — the pitching rubber, foul lines, and (during slides) players' legs and uniforms at shallow angles. Two practical mitigations, both used below:

1. **Confidence ≥ 0.5** — rubber/line hits rarely score above ~0.35 on typical broadcast frames.
2. **Aspect-ratio filter (width/height ≤ 4.2)** — bases at broadcast camera angles stay under ~4:1 (measured max 3.9:1); rubber, lines, and limbs produce thin boxes in the 4.5–7:1 range.

In [ ]:
import cv2
from pathlib import Path

# On Google Colab, uncomment to get an upload prompt:
# from google.colab import files
# uploaded = files.upload()
# INPUT_PATH = next(iter(uploaded))

INPUT_PATH = "example.jpg"   # any image or .mp4 — try a broadcast clip of your own
CONF = 0.5                   # rubber/line false positives rarely exceed ~0.35
AR_MAX = 4.2                 # bases measured up to 3.9:1 at broadcast angles; thin FPs start ~4.5:1

def keep_box(x1, y1, x2, y2):
    """Aspect-ratio filter: drop thin boxes (rubber, foul lines, limbs)."""
    w, h = x2 - x1, y2 - y1
    return h > 0 and w / h <= AR_MAX

is_video = Path(INPUT_PATH).suffix.lower() in {".mp4", ".mov", ".avi", ".mkv"}

if is_video:
    results = model.predict(INPUT_PATH, conf=CONF, stream=True, verbose=False)
    out_path = "annotated_" + Path(INPUT_PATH).stem + ".mp4"
    writer = None
    for r in results:
        frame = r.orig_img.copy()
        for b in r.boxes:
            x1, y1, x2, y2 = b.xyxy[0].tolist()
            if not keep_box(x1, y1, x2, y2):
                continue
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (255, 90, 0), 2)
            cv2.putText(frame, f"base {float(b.conf):.2f}", (int(x1), int(y1) - 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
            cv2.putText(frame, f"base {float(b.conf):.2f}", (int(x1), int(y1) - 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 90, 0), 1, cv2.LINE_AA)
        if writer is None:
            h, w = frame.shape[:2]
            writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"),
                                     30, (w, h))
        writer.write(frame)
    writer.release()
    print(f"Annotated video saved to: {out_path}")
    if "google.colab" in str(get_ipython()):
        from google.colab import files as colab_files
        colab_files.download(out_path)
else:
    result = model.predict(INPUT_PATH, conf=CONF, verbose=False)[0]
    frame = result.orig_img.copy()
    n_kept = 0
    for b in result.boxes:
        x1, y1, x2, y2 = b.xyxy[0].tolist()
        if not keep_box(x1, y1, x2, y2):
            continue
        n_kept += 1
        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (255, 90, 0), 2)
        cv2.putText(frame, f"base {float(b.conf):.2f}", (int(x1), int(y1) - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(frame, f"base {float(b.conf):.2f}", (int(x1), int(y1) - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 90, 0), 1, cv2.LINE_AA)
    plt.figure(figsize=(12, 6))
    plt.imshow(frame[:, :, ::-1])
    plt.axis("off")
    plt.title(f"{n_kept} base(s) detected at conf >= {CONF}, w/h <= {AR_MAX}")
    plt.show()

In [ ]:
yolo_dir = load_tools.load_dataset('base_detect_dataset')
print("Dataset extracted to:", yolo_dir)

with open(os.path.join(str(yolo_dir), "data.yaml")) as f:
    print("--- data.yaml ---")
    print(f.read())

n_imgs = sum(len(fs) for _, _, fs in os.walk(os.path.join(str(yolo_dir), "images")))
n_lbls = sum(len(fs) for _, _, fs in os.walk(os.path.join(str(yolo_dir), "labels")))
print(f"images: {n_imgs} | label files: {n_lbls} | unlabeled: {n_imgs - n_lbls}")

## Wrap-up

If everything above ran cleanly, the full integration works:

| Asset | Alias | Status |
|---|---|---|
| Base detector model | `base_detector` | ✅ loads via BDL API |
| COCO dataset | `base_detector_coco` | ✅ downloads + extracts |
| YOLO dataset | `base_detect_dataset` | ✅ downloads + extracts |

Next steps: resolve the 107 unlabeled images, add a val split + metrics.